# Ton, Klang, Geräusch

Demonstration der Unterschiede zwischen reinem Ton,
Klang (Summe von Harmonischen), Geräusch, Rauschen
und Impuls – jeweils im Zeit- und Frequenzbereich.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoeverc/laermarmesysteme_medien/blob/main/Python/VL1/TonKlangGeraeusch.ipynb)

In [ ]:
import os
import urllib.request

GITHUB_RAW=(
  'https://raw.githubusercontent.com'
  '/hoeverc/laermarmesysteme_medien/main')

def download(url, path):
  """Download file if not already present."""
  os.makedirs(
    os.path.dirname(path) or '.', exist_ok=True)
  if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)
    print(f'Heruntergeladen: {path}')

download(
  f'{GITHUB_RAW}/VL1/Medien'
  '/344269__inspectorj__glass-smash-bottle-h.wav',
  'Medien'
  '/344269__inspectorj__glass-smash-bottle-h.wav')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from IPython.display import Audio, display

# ===========================================================
# Einstellungen
# ===========================================================
fontsize=12
linewidth=2
freqlims=[16, 2000]
T_plot=0.02

# ===========================================================
# Signalerzeugung
# ===========================================================
f1=293.7  # Frequenz Piano D4, Hz
fs=8192   # Abtastrate, Hz
T_sig=2   # Signaldauer, s
n=T_sig*fs
t=np.arange(n)/fs
fmax=fs/2
f=np.linspace(0, fmax, n//2+1)

# --- Ton mit 3 Harmonischen ---
x_ton=np.zeros((3, n))
for ii in range(3):
  x_ton[ii, :]=(
    1/(ii+1)
    *np.cos(2*np.pi*(ii+1)*f1*t))

X_ton=np.fft.fft(x_ton, axis=1)/n
X_ton=2*X_ton[:, :n//2+1]
X_ton[:, 0]/=2
X_ton[:, -1]/=2

# --- Klang (Summe der Harmonischen) ---
x_klang=np.sum(x_ton, axis=0)
X_klang=np.fft.fft(x_klang)/n
X_klang=2*X_klang[:n//2+1]
X_klang[0]/=2
X_klang[-1]/=2

# --- Weisses Rauschen ---
rng=np.random.default_rng(42)
x_rauschen=-1+2*rng.random(n)
X_rauschen=np.fft.fft(x_rauschen)/n
X_rauschen=2*X_rauschen[:n//2+1]
X_rauschen[0]/=2
X_rauschen[-1]/=2

# --- Geraeusch (Audiodatei) ---
sr2, data2=wavfile.read(
  'Medien'
  '/344269__inspectorj__glass-smash-bottle-h.wav')
if data2.ndim>1:
  data2=data2[:, 0]
if np.issubdtype(data2.dtype, np.integer):
  max_val=np.iinfo(data2.dtype).max
  x_geraeusch=data2.astype(np.float64)/max_val
else:
  x_geraeusch=data2.astype(np.float64)
fs2=sr2
n2=len(x_geraeusch)
t2=np.arange(n2)/fs2
fmax2=fs2/2
f2=np.linspace(0, fmax2, n2//2+1)
X_geraeusch=np.fft.fft(x_geraeusch)/n2
X_geraeusch=2*X_geraeusch[:n2//2+1]
X_geraeusch[0]/=2
X_geraeusch[-1]/=2

# --- Impuls ---
x_imp_raw=x_ton[2, :]*np.exp(-fs/4*t)
ind=np.argmax(x_imp_raw<=0)
x_impuls=np.concatenate([
  np.zeros(10),
  x_imp_raw[ind-1::-1],
  [1.0],
  x_imp_raw[:n-ind-11]])
X_impuls=np.fft.fft(x_impuls)/n
X_impuls=2*X_impuls[:n//2+1]
X_impuls[0]/=2
X_impuls[-1]/=2

## Reiner Ton (D4)

In [ ]:
fig, (ax1, ax2)=plt.subplots(
  1, 2, figsize=(12, 4))

ax1.plot(t, x_ton[0, :], linewidth=linewidth)
ax1.set_xlabel(r'$t$ in s', fontsize=fontsize)
ax1.set_ylabel(
  r'$x(t)$ in a.u.', fontsize=fontsize)
ax1.set_xlim(0, T_plot)
ax1.set_ylim(-1.1, 1.1)
ax1.set_title('Reiner Ton (D4) - Zeitsignal')
ax1.tick_params(labelsize=fontsize)

ax2.plot(
  f, np.abs(X_ton[0, :]), linewidth=linewidth)
ax2.set_xlabel(r'$f$ in Hz', fontsize=fontsize)
ax2.set_ylabel(
  r'$X(f)$ in a.u.', fontsize=fontsize)
ax2.set_xlim(*freqlims)
ax2.set_ylim(0, 1.1)
ax2.set_title(
  'Reiner Ton (D4) - Frequenzspektrum')
ax2.tick_params(labelsize=fontsize)

plt.tight_layout()
plt.show()

display(Audio(data=x_ton[0, :], rate=fs))

## Klang (Harmonische)

In [ ]:
fig, axes=plt.subplots(4, 2, figsize=(12, 12))

for ii in range(3):
  axes[ii, 0].plot(
    t, x_ton[ii, :], linewidth=linewidth)
  axes[ii, 0].set_xlabel(
    r'$t$ in s', fontsize=fontsize)
  axes[ii, 0].set_ylabel(
    r'$x(t)$ in a.u.', fontsize=fontsize)
  axes[ii, 0].set_xlim(0, T_plot)
  axes[ii, 0].set_ylim(-1.1, 1.1)
  axes[ii, 0].set_title(
    f'{ii+1}. Harmonische (D4) - Zeitsignal')
  axes[ii, 0].tick_params(labelsize=fontsize)

  axes[ii, 1].plot(
    f, np.abs(X_ton[ii, :]),
    linewidth=linewidth)
  axes[ii, 1].set_xlabel(
    r'$f$ in Hz', fontsize=fontsize)
  axes[ii, 1].set_ylabel(
    r'$X(f)$ in a.u.', fontsize=fontsize)
  axes[ii, 1].set_xlim(*freqlims)
  axes[ii, 1].set_ylim(0, 1.1)
  axes[ii, 1].set_title(
    f'{ii+1}. Harmonische (D4) - '
    'Frequenzspektrum')
  axes[ii, 1].tick_params(labelsize=fontsize)

# Klang (Summe)
axes[3, 0].plot(
  t, x_klang, linewidth=linewidth)
axes[3, 0].set_xlabel(
  r'$t$ in s', fontsize=fontsize)
axes[3, 0].set_ylabel(
  r'$x(t)$ in a.u.', fontsize=fontsize)
axes[3, 0].set_xlim(0, T_plot)
kl_max=np.max(np.abs(x_klang))
axes[3, 0].set_ylim(-1.1*kl_max, 1.1*kl_max)
axes[3, 0].set_title('Klang - Zeitsignal')
axes[3, 0].tick_params(labelsize=fontsize)

axes[3, 1].plot(
  f, np.abs(X_klang), linewidth=linewidth)
axes[3, 1].set_xlabel(
  r'$f$ in Hz', fontsize=fontsize)
axes[3, 1].set_ylabel(
  r'$X(f)$ in a.u.', fontsize=fontsize)
axes[3, 1].set_xlim(*freqlims)
axes[3, 1].set_ylim(
  0, 1.1*np.max(np.abs(X_klang)))
axes[3, 1].set_title(
  'Klang - Frequenzspektrum')
axes[3, 1].tick_params(labelsize=fontsize)

plt.tight_layout()
plt.show()

# Audio: Harmonische einzeln und Klang
for ii in range(3):
  print(f'{ii+1}. Harmonische:')
  display(Audio(data=x_ton[ii, :], rate=fs))
print('Klang (Summe):')
display(Audio(data=x_klang, rate=fs))

## Geräusch und Rauschen

In [ ]:
fig, axes=plt.subplots(2, 2, figsize=(12, 8))

# Geraeusch (zerbrechendes Glas)
axes[0, 0].plot(
  t2, x_geraeusch, linewidth=linewidth)
axes[0, 0].set_xlabel(
  r'$t$ in s', fontsize=fontsize)
axes[0, 0].set_ylabel(
  r'$x(t)$ in a.u.', fontsize=fontsize)
axes[0, 0].set_xlim(0, 1.5)
axes[0, 0].set_ylim(-1.1, 1.1)
axes[0, 0].set_title(
  'Geraeusch (zerbr. Glas) - Zeitsignal')
axes[0, 0].tick_params(labelsize=fontsize)

axes[0, 1].plot(
  f2, np.abs(X_geraeusch),
  linewidth=linewidth)
axes[0, 1].set_xlabel(
  r'$f$ in Hz', fontsize=fontsize)
axes[0, 1].set_ylabel(
  r'$X(f)$ in a.u.', fontsize=fontsize)
axes[0, 1].set_xlim(100, 4000)
axes[0, 1].set_title(
  'Geraeusch (zerbr. Glas) - '
  'Frequenzspektrum')
axes[0, 1].tick_params(labelsize=fontsize)

# Weisses Rauschen
axes[1, 0].plot(
  t, x_rauschen, linewidth=linewidth)
axes[1, 0].set_xlabel(
  r'$t$ in s', fontsize=fontsize)
axes[1, 0].set_ylabel(
  r'$x(t)$ in a.u.', fontsize=fontsize)
axes[1, 0].set_xlim(0, 1.5)
axes[1, 0].set_ylim(-1.1, 1.1)
axes[1, 0].set_title(
  'Rauschen - Zeitsignal')
axes[1, 0].tick_params(labelsize=fontsize)

axes[1, 1].plot(
  f, np.abs(X_rauschen),
  linewidth=linewidth)
axes[1, 1].set_xlabel(
  r'$f$ in Hz', fontsize=fontsize)
axes[1, 1].set_ylabel(
  r'$X(f)$ in a.u.', fontsize=fontsize)
axes[1, 1].set_xlim(100, 4000)
axes[1, 1].set_title(
  'Rauschen - Frequenzspektrum')
axes[1, 1].tick_params(labelsize=fontsize)

plt.tight_layout()
plt.show()

print('Geraeusch (zerbrechendes Glas):')
display(Audio(data=x_geraeusch, rate=fs2))
print('Weisses Rauschen:')
display(Audio(data=x_rauschen, rate=fs))

## Impuls

In [ ]:
fig, (ax1, ax2)=plt.subplots(
  1, 2, figsize=(12, 4))

ax1.plot(t, x_impuls, linewidth=linewidth)
ax1.set_xlabel(r'$t$ in s', fontsize=fontsize)
ax1.set_ylabel(
  r'$x(t)$ in a.u.', fontsize=fontsize)
ax1.set_xlim(0, T_plot)
ax1.set_ylim(-1.1, 1.1)
ax1.set_title('Impuls - Zeitsignal')
ax1.tick_params(labelsize=fontsize)

ax2.plot(
  f, np.abs(X_impuls), linewidth=linewidth)
ax2.set_xlabel(r'$f$ in Hz', fontsize=fontsize)
ax2.set_ylabel(
  r'$X(f)$ in a.u.', fontsize=fontsize)
ax2.set_xlim(*freqlims)
ax2.set_title('Impuls - Frequenzspektrum')
ax2.tick_params(labelsize=fontsize)

plt.tight_layout()
plt.show()

display(Audio(data=x_impuls, rate=fs))